# Veridelta Quickstart

Veridelta is a high-performance semantic diffing engine built on Polars. Standard binary equality checks often fail in production pipelines due to trivial transformation artifacts like floating-point noise, rounding algorithm mismatches, or string formatting variance. 

Veridelta enables data teams to define **logical equality** through declarative configuration, executing memory-safe validations at scale via zero-copy lazy evaluation.

## 1. The Challenge: Physical vs. Logical Equality

Data migration between disparate computational environments (e.g., Python application layers to SQL data warehouses) introduces expected transformation artifacts. Two common examples include:

1. **Rounding Algorithm Mismatches:** Python utilizes "Banker's Rounding" (round half to even; `10.445` evaluates to `10.44`). SQL systems typically utilize standard rounding (round half away from zero; `10.445` evaluates to `10.45`). This generates an artificial `$0.01` variance.
2. **String Casing & Padding:** Legacy systems may export Title Case (`"Pending"`), while modern warehouses enforce standardized, padded UPPERCASE (`" PENDING "`).

These datasets are logically identical, yet standard physical equality checks fail instantly.

In [ ]:
import polars as pl

import veridelta as vd
from veridelta.engine import DiffEngine
from veridelta.models import DiffConfig, DiffRule

# Source System (Legacy Python - Banker's Rounding, Title Case)
src_df = pl.DataFrame(
    {
        "transaction_id": ["TXN-001", "TXN-002", "TXN-003"],
        "tax_amount": [10.44, 250.50, 300.00],  # 10.445 rounded half-to-even
        "status": ["Cleared", "Pending", "Failed"],
    }
)

# Target System (Modern SQL - Standard Rounding, UPPERCASE with padding)
tgt_df = pl.DataFrame(
    {
        "transaction_id": ["TXN-001", "TXN-002", "TXN-003"],
        "tax_amount": [10.45, 250.50, 300.00],  # 10.445 rounded half-up
        "status": [" CLEARED ", " PENDING ", " FAILED "],
    }
)

is_physically_equal = src_df.equals(tgt_df)
print(f"Strict Physical Equality: {is_physically_equal}")

# Output:
# Strict Physical Equality: False

## 2. Defining Semantic Boundaries

Veridelta resolves physical variance by establishing semantic boundaries. The `DiffConfig` object declaratively defines acceptable tolerances and normalization rules, masking expected migration artifacts without ignoring genuine data corruption.

*Note: Veridelta is optimized for out-of-core processing. We pass `.lazy()` frames to the engine to bypass memory constraints.*

In [ ]:
# Define the semantic comparison boundaries
config = DiffConfig(
    primary_keys=["transaction_id"],
    # Global Policies: Applied to all columns automatically
    default_absolute_tolerance=0.01,  # Allow a $0.01 variance for Banker's Rounding
    default_whitespace_mode="both",  # Strip all leading/trailing whitespace
    rules=[
        DiffRule(
            column_names=["status"],
            case_insensitive=True,  # Ensure 'Pending' matches 'PENDING'
        )
    ],
)

# Initialize the engine and evaluate the lazy computation graphs
engine = DiffEngine(config)
summary = engine.compare_lazyframes(src_df.lazy(), tgt_df.lazy())

print(summary.report_summary)

# Output:
# Veridelta Execution Summary
# ===========================
# Status:        PASSED (Perfect Match)
# Match Rate:    100.0%
# Source Rows:   3
# Target Rows:   3
# Volume Shift:  +0 rows
#
# Row-Level Discrepancies:
# ---------------------------
# Added:         0
# Removed:       0
# Changed:       0
# Total Issues:  0

## 3. Scale & Performance: Simulating Pipeline Drift

Veridelta is built entirely on Polars' relational algebra. It does not iterate over rows. 

The following benchmark loads a sample of the NYC Yellow Taxi dataset instantly as a lazy computation graph, injects programmatic ETL drift, and executes a multi-threaded semantic comparison without ever fully loading the datasets into RAM.

In [ ]:
# load_nyc_taxi() returns a Polars LazyFrame, keeping memory footprint at zero
src_lf = vd.datasets.load_nyc_taxi().unique(  # pyright: ignore[reportUnknownMemberType]
    subset=["VendorID", "tpep_pickup_datetime"]
)

# Simulate ETL drift on the target system:
# - Rounding noise on the fare (+$0.002)
# - A 0.5% calculation fluctuation on the tip amount (e.g., exchange rate drift)
# - Inconsistent string formatting
tgt_lf = src_lf.with_columns(  # pyright: ignore[reportUnknownMemberType]
    fare_amount=pl.col("fare_amount") + 0.002,
    tip_amount=pl.col("tip_amount") * 1.005,
    store_and_fwd_flag=pl.col("store_and_fwd_flag").str.to_uppercase(),
)

config = DiffConfig(
    primary_keys=["VendorID", "tpep_pickup_datetime"],
    default_absolute_tolerance=0.01,
    rules=[
        DiffRule(column_names=["store_and_fwd_flag"], case_insensitive=True),
        DiffRule(
            column_names=["tip_amount"],
            relative_tolerance=0.01,  # Accept up to 1% relative drift on tips
        ),
    ],
)

engine = DiffEngine(config)
summary = engine.compare_lazyframes(src_lf, tgt_lf)

print(summary.report_summary)

# Output:
# Veridelta Execution Summary
# ===========================
# Status:        PASSED (Perfect Match)
# Match Rate:    100.0%
# Source Rows:   908
# Target Rows:   908
# Volume Shift:  +0 rows
#
# Row-Level Discrepancies:
# ---------------------------
# Added:         0
# Removed:       0
# Changed:       0
# Total Issues:  0

## Next Steps & Case Studies

This quickstart covers basic mathematical tolerances and string normalization. To master Veridelta, explore the core tutorials or dive into specific industry case studies.

### Core Tutorials
* **[CLI Quickstart](quickstart_cli.ipynb):** Learn how to execute file-based comparisons directly from your terminal.
* **[YAML Configuration](yaml_config.ipynb):** Learn how to drive Veridelta entirely via declarative YAML files for CI/CD pipelines.
* **[Advanced Rules](advanced_rules.ipynb):** Learn how to utilize Regex replacements, value crosswalks, and strict schema enforcement modes.

### Industry Case Studies
* **[Case Study: Financial Ledger Reconciliation](financial_ledger.ipynb)** - Handling complex categorical crosswalks, negative value coercion, and missing column handling.
* **[Case Study: Scientific Data Sanitization](scientific_data_migration.ipynb):** Resolving legacy integer bugs, enforcing physical law constraints, and isolating intentional business logic using the Two-Gate Validation Architecture.